In [7]:
import sqlite3
import os
import sys
import time
import numpy as np
from astropy.io import fits
import glob
from tqdm import tqdm
import pandas as pd

In [9]:
def get_connection(db_file, timeout=30, retries=30, retry_delay=0.5):
    for attempt in range(retries):
        try:
            conn = sqlite3.connect(db_file, timeout=timeout)
            conn.execute('PRAGMA busy_timeout=30000;')
            return conn
        except sqlite3.OperationalError as e:
            if 'locked' in str(e).lower() and attempt < retries - 1:
                time.sleep(retry_delay)
                continue
            print(f"Database connection failed: {e}")
            return None
        except sqlite3.Error as e:
            print(f"Database connection failed: {e}")
            return None
    return None

def create_table(conn, create_table_sql, retries=20, retry_delay=0.5):
    if conn is None:
        raise RuntimeError("No database connection available to create table.")
    c = conn.cursor()
    for attempt in range(retries):
        try:
            c.execute(create_table_sql)
            return True
        except sqlite3.OperationalError as e:
            if 'locked' in str(e).lower() and attempt < retries - 1:
                time.sleep(retry_delay)
                continue
            raise
    return False

keys = [
    'OBJECT',      # target designation
    'RA',          # right ascension (deg)
    'DEC',         # declination (deg)
    'EXPTIME',     # total integration time (s)
    'MJD-OBS',     # observation start (d)
    'MJD-END',     # observation end (d)
    'WAVELMIN',    # min wavelength (nm)
    'WAVELMAX',    # max wavelength (nm)
    'SPEC_BIN',    # spectral bin size (nm)
    'SNR',         # signal-to-noise per pixel
    'SPEC_RES'     # spectral resolution
]

typ = [
    'text',
    'float',
    'float',
    'float',
    'float',
    'float',
    'float',
    'float',
    'float',
    'float',
    'float'
]

sql = "CREATE TABLE IF NOT EXISTS spectra ("
for i, (k, t) in enumerate(zip(keys, typ)):
    if i == 0:
        sql += '{} {} PRIMARY KEY'.format(k.replace('-','_').replace(' ','_'), t)
    else:
        sql += ',{} {}'.format(k.replace('-','_').replace(' ','_'), t)
sql += ');'

path = '/home/msp25gd/Downloads/'
fs = glob.glob(f'{path}ADP*')

print('using path', path)
print('found', len(fs), 'files')
if len(fs) > 0:
    print(fs[:20])

assert len(keys) == len(typ), "keys/typ length mismatch"

conn = get_connection("spectra.db")
if conn is None:
    raise RuntimeError("Unable to open spectra.db")

with conn:
    cur = conn.cursor()
    cur.execute('PRAGMA journal_mode=WAL;')
    cur.execute('PRAGMA busy_timeout=30000;')

    create_table(conn, sql)

    rowcount = 0
    insert_sql = "INSERT OR IGNORE INTO spectra ({}) VALUES({})".format(
        ','.join(k.replace('-','_').replace(' ','_') for k in keys),
        ','.join('?' for _ in keys)
    )

    for file in tqdm(fs):
        try:
            with fits.open(file) as h:
                val = []
                for k in keys:
                    try:
                        val.append(h[0].header[k])
                    except Exception:
                        val.append(None)
        except Exception as e:
            print("Could not read", file, ":", e)
            continue

        for attempt in range(20):
            try:
                cur.execute(insert_sql, val)
                break
            except sqlite3.OperationalError as e:
                if 'locked' in str(e).lower() and attempt < 19:
                    time.sleep(0.5)
                else:
                    print("INSERT failed for", file, ":", e)
                    break

        rowcount += 1
        if rowcount % 5000 == 0:
            conn.commit()

    conn.commit()

conn.close()

using path /home/msp25gd/Downloads/
found 84188 files
['/home/msp25gd/Downloads/ADP.2020-06-09T06:16:46.190', '/home/msp25gd/Downloads/ADP.2021-08-31T16:27:33.087', '/home/msp25gd/Downloads/ADP.2024-03-27T09:00:53.263', '/home/msp25gd/Downloads/ADP.2020-06-19T09:47:57.146', '/home/msp25gd/Downloads/ADP.2020-07-17T10:15:23.409', '/home/msp25gd/Downloads/ADP.2020-06-15T07:06:30.083', '/home/msp25gd/Downloads/ADP.2022-02-03T13:06:05.792', '/home/msp25gd/Downloads/ADP.2022-11-22T09:54:52.265', '/home/msp25gd/Downloads/ADP.2020-07-10T21:47:17.298', '/home/msp25gd/Downloads/ADP.2020-06-19T08:52:52.320', '/home/msp25gd/Downloads/ADP.2020-08-12T09:24:16.886', '/home/msp25gd/Downloads/ADP.2023-09-25T11:31:33.118', '/home/msp25gd/Downloads/ADP.2021-09-09T11:25:13.997', '/home/msp25gd/Downloads/ADP.2020-06-09T14:14:31.669', '/home/msp25gd/Downloads/ADP.2020-06-15T11:55:46.201', '/home/msp25gd/Downloads/ADP.2023-06-06T12:31:23.007', '/home/msp25gd/Downloads/ADP.2021-08-29T12:18:51.354', '/home/msp

100%|██████████| 84188/84188 [49:56<00:00, 28.09it/s]  


In [10]:
# verify how many rows ended up in the table
conn = get_connection("spectra.db")
with conn:
    cur = conn.cursor()
    cur.execute("SELECT COUNT(*) FROM spectra")
    print("rows in spectra:", cur.fetchone()[0])

rows in spectra: 18088


In [11]:
conn = sqlite3.connect("spectra.db")
df = pd.read_sql_query("SELECT * FROM spectra LIMIT 10", conn)
conn.close()

display(df)

,OBJECT,RA,DEC,EXPTIME,MJD_OBS,MJD_END,WAVELMIN,WAVELMAX,SPEC_BIN,SNR,SPEC_RES
0,J1032+0927,158.088363,9.46379,2129.0002,58928.107738,58928.132379,565.503700,946.378300,0.003500,4.381700,42310.0
1,HD_137753,232.493530,-52.30335,300.0003,52685.370872,52685.374344,472.684000,683.503200,0.001400,529.047300,74450.0
2,HD151852,252.303028,19.29439,1099.9993,60381.365996,60381.378728,472.644700,683.498700,0.001400,626.448900,74450.0
3,HD17652,42.272332,-32.40455,4.6984,55434.317453,55434.317508,664.991000,1042.600000,0.002000,140.122400,107200.0
4,SDSS-J013901.40-082443,24.756041,-8.41191,4800.0000,53286.268257,53286.323812,472.688700,683.506300,0.002800,8.458100,42310.0
5,HIP113086,343.530857,19.89189,480.0038,55784.192984,55784.199642,328.189403,456.294329,0.001367,112.820501,53750.0
6,Q0439-433,70.321883,-43.22912,3420.0009,52556.268358,52556.307941,373.116400,499.920500,0.003000,28.169400,40970.0
7,HD 284248,63.650490,22.34940,3000.0001,59858.222586,59858.257308,328.197400,456.299000,0.001300,195.799400,58640.0
8,HD2796,7.819779,-16.79436,300.0016,53693.027493,53693.030966,373.213300,499.978600,0.001500,150.802500,71050.0
9,CoRoT-9b,280.786295,6.20432,2399.9965,55365.094635,55365.122985,495.917267,707.082286,0.001482,26.522872,66320.0
